# Parte 5 — Questões de negócio: Nível 4 (casos críticos)

**Q11 e Q12.** Acidentes com 3 ou mais mortos e o ranking dos municípios de
Pernambuco mais afetados em 2024 e 2025.

Mesmas regras dos níveis anteriores: taxa de letalidade = % de acidentes fatais do
grupo, comparada à taxa global de **7,16%**, volume mínimo declarado em cada
filtro e leitura sem linguagem causal.


## 0. Base analítica da Parte 4

Lida de `dados_tratados/base_analitica_prf_2023_2025.csv` — a base já vem no
repositório. Se o arquivo não existir, rode antes o notebook da Parte 4.


In [1]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd


def localizar_raiz() -> Path:
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "dados_brutos").is_dir():
            return candidato
    raise FileNotFoundError("Abra o notebook da raiz do projeto ou de 'notebooks/'.")


RAIZ = localizar_raiz()
DIR_BRUTOS = RAIZ / "dados_brutos"
DIR_TRATADOS = RAIZ / "dados_tratados"
DIR_RESULTADOS = RAIZ / "resultados"
for pasta in (DIR_TRATADOS, DIR_RESULTADOS):
    pasta.mkdir(parents=True, exist_ok=True)

SEPARADOR = ";"
ENCODING_SAIDA = "utf-8-sig"
ANOS = (2023, 2024, 2025)
print(RAIZ)


/home/user/PRF-pandas-e-Plotly---2023-2025


In [2]:
ARQ_ANALITICA = DIR_TRATADOS / "base_analitica_prf_2023_2025.csv"
df = pd.read_csv(ARQ_ANALITICA, sep=SEPARADOR, encoding=ENCODING_SAIDA,
                 dtype={"id": "string"}, low_memory=False)
df["data_inversa"] = pd.to_datetime(df["data_inversa"], errors="coerce")

TAXA_GLOBAL = df["acidente_fatal"].mean()
print(f"Base: {df.shape[0]:,} acidentes | taxa global de fatais: {TAXA_GLOBAL:.4%}")


Base: 213,497 acidentes | taxa global de fatais: 7.1631%


In [3]:
# --- funções de apoio usadas nas questões ---
def indicadores(base: pd.DataFrame, coluna: str, minimo: int = 0) -> pd.DataFrame:
    """Acidentes, fatais, mortos e taxa de letalidade por categoria.

    Neste exercício, taxa de letalidade = % de acidentes fatais do grupo
    (definição do enunciado), comparada com a taxa global.
    """
    tab = base.groupby(coluna).agg(
        acidentes=("acidente_fatal", "size"),
        fatais=("acidente_fatal", "sum"),
        mortos=("mortos", "sum"),
    )
    tab = tab[tab["acidentes"] >= minimo]
    tab["taxa_letalidade"] = tab["fatais"] / tab["acidentes"]
    tab["lift"] = tab["taxa_letalidade"] / TAXA_GLOBAL
    return tab.sort_values("taxa_letalidade", ascending=False)


def fmt(tab: pd.DataFrame) -> pd.DataFrame:
    """Formata percentuais para leitura (ignora colunas ausentes)."""
    saida = tab.copy()
    if "taxa_letalidade" in saida.columns:
        saida["taxa_letalidade"] = saida["taxa_letalidade"].map("{:.2%}".format)
    if "lift" in saida.columns:
        saida["lift"] = saida["lift"].map("{:.2f}".format)
    return saida


---
## Q11 — Acidentes críticos: 3 ou mais mortos.
Quantos são, **qual UF lidera** e qual a principal causa? O perfil completo vem a seguir.
Volume mínimo: não se aplica — o filtro é o critério do enunciado (3 ou mais mortos),
sem corte de volume: os **437** acidentes atendem ao critério.


In [4]:
criticos = df[df["mortos"] >= 3].copy()
fatais = df[df["acidente_fatal"] == 1]

print(f"Acidentes com 3 ou mais mortos : {len(criticos):,}")
print(f"Acidentes fatais (1 ou mais)   : {len(fatais):,}")
print(f"Mortes no período              : {int(df['mortos'].sum()):,}")
print()
print(f"São {len(criticos) / len(df):.3%} de todos os acidentes e "
      f"{len(criticos) / len(fatais):.2%} dos acidentes fatais,")
print(f"mas concentram {int(criticos['mortos'].sum()):,} mortes "
      f"({criticos['mortos'].sum() / df['mortos'].sum():.2%} do total).")
print()
print(f"Média de mortos por acidente crítico: {criticos['mortos'].mean():.2f} "
      f"(mediana {int(criticos['mortos'].median())})")
print(f"Média de mortos nos demais fatais   : "
      f"{fatais.loc[fatais['mortos'] < 3, 'mortos'].mean():.2f}")

dist = criticos.groupby("mortos").agg(acidentes=("id", "size"),
                                      mortos=("mortos", "sum"))
print("Distribuição dos acidentes críticos:")
display(dist.head(8))

assert len(criticos) == 437, f"Esperava 437 críticos, veio {len(criticos)}"
assert (criticos["mortos"] >= 3).all()
assert criticos["acidente_fatal"].all(), "Todo acidente crítico é fatal"


Acidentes com 3 ou mais mortos : 437
Acidentes fatais (1 ou mais)   : 15,293
Mortes no período              : 17,833

São 0.205% de todos os acidentes e 2.86% dos acidentes fatais,
mas concentram 1,669 mortes (9.36% do total).

Média de mortos por acidente crítico: 3.82 (mediana 3)
Média de mortos nos demais fatais   : 1.09
Distribuição dos acidentes críticos:


,acidentes,mortos
mortos,,
3,266,798
4,104,416
5,39,195
6,12,72
7,6,42
8,1,8
9,3,27
10,1,10


In [5]:
# qual UF lidera os acidentes críticos
q11_uf = criticos.groupby("uf").agg(
    acidentes=("id", "size"),
    mortos=("mortos", "sum"),
).sort_values("acidentes", ascending=False)
display(q11_uf.head(8))

lidera = q11_uf.index[0]
print(f"UF que mais concentra: {lidera} — "
      f"{int(q11_uf['acidentes'].iloc[0])} acidentes "
      f"({q11_uf['acidentes'].iloc[0] / len(criticos):.1%} dos críticos) e "
      f"{int(q11_uf['mortos'].iloc[0])} mortos "
      f"({q11_uf['mortos'].iloc[0] / criticos['mortos'].sum():.1%} das mortes críticas)")
print(f"UFs com acidente crítico: {len(q11_uf)}")

assert lidera == "MG", f"Esperava MG na liderança, veio {lidera}"
assert q11_uf["acidentes"].iloc[0] >= q11_uf["acidentes"].iloc[1]


,acidentes,mortos
uf,,
MG,75,303
BA,68,289
PR,42,155
GO,32,121
MT,30,122
RS,25,88
SC,24,79
MS,18,60


UF que mais concentra: MG — 75 acidentes (17.2% dos críticos) e 303 mortos (18.2% das mortes críticas)
UFs com acidente crítico: 27


In [6]:
def composicao(criticos_df: pd.DataFrame, fatais_df: pd.DataFrame,
               coluna: str) -> pd.DataFrame:
    """Participação de cada categoria entre os críticos e entre todos os fatais.

    razão = % dos críticos / % dos fatais. Razão acima de 1 significa
    categoria sobre-representada entre os acidentes críticos.
    """
    a = criticos_df.groupby(coluna).size().rename("criticos")
    b = fatais_df.groupby(coluna).size().rename("fatais")
    comp = pd.concat([a, b], axis=1).fillna(0)
    comp["pct_criticos"] = comp["criticos"] / comp["criticos"].sum() * 100
    comp["pct_fatais"] = comp["fatais"] / comp["fatais"].sum() * 100
    comp["razao"] = comp["pct_criticos"] / comp["pct_fatais"]
    return comp.sort_values("criticos", ascending=False)


q11_tipo = composicao(criticos, fatais, "tipo_acidente")
print("Tipo de acidente — composição dos críticos x dos fatais:")
display(q11_tipo[q11_tipo["criticos"] >= 3][
    ["criticos", "pct_criticos", "pct_fatais", "razao"]
].round(2).head(10))

frontal = q11_tipo.loc["Colisão frontal"]
print(f"Colisão frontal: {int(frontal['criticos'])} dos {len(criticos)} críticos "
      f"({frontal['pct_criticos']:.1f}%) contra {frontal['pct_fatais']:.1f}% dos fatais "
      f"— razão {frontal['razao']:.2f}")

assert frontal["razao"] > 1.5, "A frontal deveria sobressair nos críticos"


Tipo de acidente — composição dos críticos x dos fatais:


,criticos,pct_criticos,pct_fatais,razao
tipo_acidente,,,,
Colisão frontal,275.0,62.93,27.74,2.27
Saída de leito carroçável,48.0,10.98,11.85,0.93
Colisão transversal,24.0,5.49,8.06,0.68
Colisão traseira,23.0,5.26,11.30,0.47
Colisão lateral sentido oposto,22.0,5.03,3.54,1.42
Engavetamento,13.0,2.97,0.61,4.84
Colisão com objeto,13.0,2.97,5.13,0.58
Tombamento,9.0,2.06,5.07,0.41
Colisão lateral mesmo sentido,6.0,1.37,3.86,0.36


Colisão frontal: 275 dos 437 críticos (62.9%) contra 27.7% dos fatais — razão 2.27


In [7]:
q11_causa = composicao(criticos, fatais, "causa_acidente")
print("Causa do acidente — composição dos críticos x dos fatais:")
display(q11_causa[q11_causa["criticos"] >= 5][
    ["criticos", "pct_criticos", "pct_fatais", "razao"]
].round(2).head(12))

contramao = q11_causa.loc["Transitar na contramão"]
ultrapassagem = q11_causa.loc["Ultrapassagem Indevida"]
print(f"Contramão + ultrapassagem indevida: "
      f"{int(contramao['criticos'] + ultrapassagem['criticos'])} dos {len(criticos)} "
      f"críticos ({(contramao['criticos'] + ultrapassagem['criticos']) / len(criticos):.1%})")

sobrerepresentadas = q11_causa[q11_causa["criticos"] >= 10].sort_values(
    "razao", ascending=False)
print("Causas mais sobrerrepresentadas (>= 10 acidentes críticos):")
for nome, linha in sobrerepresentadas.head(3).iterrows():
    print(f"  {nome[:46]:<46} {int(linha['criticos']):>3} críticos  "
          f"razão {linha['razao']:.2f}")

assert ultrapassagem["razao"] > 1.5


Causa do acidente — composição dos críticos x dos fatais:


,criticos,pct_criticos,pct_fatais,razao
causa_acidente,,,,
Transitar na contramão,118.0,27.00,13.82,1.95
Ultrapassagem Indevida,66.0,15.10,5.53,2.73
Velocidade Incompatível,43.0,9.84,7.42,1.33
Reação tardia ou ineficiente do condutor,40.0,9.15,10.51,0.87
Ausência de reação do condutor,29.0,6.64,13.54,0.49
Condutor Dormindo,29.0,6.64,2.99,2.22
Manobra de mudança de faixa,23.0,5.26,3.71,1.42
Ingestão de álcool pelo condutor,12.0,2.75,3.30,0.83
Acessar a via sem observar a presença dos outros veículos,11.0,2.52,7.32,0.34


Contramão + ultrapassagem indevida: 184 dos 437 críticos (42.1%)
Causas mais sobrerrepresentadas (>= 10 acidentes críticos):
  Ultrapassagem Indevida                          66 críticos  razão 2.73
  Condutor Dormindo                               29 críticos  razão 2.22
  Transitar na contramão                         118 críticos  razão 1.95


In [8]:
print("Tipo de pista:")
display(composicao(criticos, fatais, "tipo_pista")[
    ["criticos", "pct_criticos", "pct_fatais", "razao"]
].round(2))

print("Fase do dia:")
display(composicao(criticos, fatais, "fase_dia")[
    ["criticos", "pct_criticos", "pct_fatais", "razao"]
].round(2))

print(f"Pessoas envolvidas — mediana nos críticos: {criticos['pessoas'].median():.0f} "
      f"| nos demais fatais: {fatais.loc[fatais['mortos'] < 3, 'pessoas'].median():.0f} "
      f"| na base toda: {df['pessoas'].median():.0f}")
print(f"Veículos envolvidos — mediana nos críticos: {criticos['veiculos'].median():.0f} "
      f"| nos demais fatais: {fatais.loc[fatais['mortos'] < 3, 'veiculos'].median():.0f}")


Tipo de pista:


,criticos,pct_criticos,pct_fatais,razao
tipo_pista,,,,
Simples,387,88.56,66.85,1.32
Dupla,42,9.61,27.59,0.35
Múltipla,8,1.83,5.56,0.33


Fase do dia:


,criticos,pct_criticos,pct_fatais,razao
fase_dia,,,,
Pleno dia,191,43.71,38.74,1.13
Plena Noite,179,40.96,48.55,0.84
Amanhecer,49,11.21,7.62,1.47
Anoitecer,18,4.12,5.09,0.81


Pessoas envolvidas — mediana nos críticos: 7 | nos demais fatais: 3 | na base toda: 2
Veículos envolvidos — mediana nos críticos: 3 | nos demais fatais: 2


In [9]:
q11_br = criticos.groupby("br").agg(acidentes=("id", "size"),
                                  mortos=("mortos", "sum")).sort_values(
    "mortos", ascending=False)
display(q11_br.head(10))

print(f"BRs com acidente crítico: {len(q11_br)}")
print(f"O top 10 concentra {int(q11_br.head(10)['mortos'].sum()):,} dos "
      f"{int(criticos['mortos'].sum()):,} mortos críticos "
      f"({q11_br.head(10)['mortos'].sum() / criticos['mortos'].sum():.1%})")

maior = criticos.nlargest(1, "mortos").iloc[0]
print()
print(f"Maior acidente crítico do período: {int(maior['mortos'])} mortos em "
      f"{maior['municipio']} ({maior['uf']}), BR-{maior['br']}, "
      f"{maior['data_inversa'].date()} — {maior['tipo_acidente']} / "
      f"{maior['causa_acidente']} ({int(maior['pessoas'])} pessoas envolvidas)")


,acidentes,mortos
br,,
116,48,207
101,31,120
163,30,114
153,23,83
40,17,63
364,17,58
242,12,49
381,11,43
20,12,43


BRs com acidente crítico: 69
O top 10 concentra 821 dos 1,669 mortos críticos (49.2%)

Maior acidente crítico do período: 37 mortos em TEOFILO OTONI (MG), BR-116, 2024-12-21 — Colisão lateral sentido oposto / Carga excessiva e/ou mal acondicionada (54 pessoas envolvidas)


**Interpretação.** Os acidentes críticos são **raros e concentrados**: 437 ocorrências
(0,205% da base) — **100% fatais por construção, contra 7,16% da global** — que respondem por
2,86% dos acidentes fatais, mas por **9,36% de todas as mortes**. A **UF que mais concentra** é
**Minas Gerais** — 75 acidentes (17,2%) e **303 das 1.669 mortes** (18,2%) — e a principal causa
é **transitar na contramão** (27,0% dos críticos).


**Detalhamento.** A distribuição é assimétrica: 266 acidentes com 3 mortos, 104 com 4 e apenas
67 com 5 ou mais, numa média de 3,8 mortos por acidente contra 1,1 nos demais fatais. Entre as
causas, **ultrapassagem indevida** tem razão **2,73** — a maior entre as causas com 10 ou mais
críticos — e `Colisão frontal` responde por **62,9%** dos críticos contra 27,7% dos fatais
(razão **2,27**).

Dois recortes de exposição completam o quadro: **pista Simples** concentra **88,6%** dos
críticos (contra 66,9% dos fatais) e **duas fases do dia ficam sobrerrepresentadas** —
**Amanhecer** (11,2% contra 7,6%, razão 1,47) e **Pleno dia** (43,7% contra 38,7%, razão
1,13) — enquanto Plena Noite, a fase mais letal depois de Amanhecer, tem razão 0,84.

A concentração por rodovia é forte: **69 BRs** registraram acidentes críticos e as 10
primeiras concentram **49,2%** das mortes críticas, com BR-116, BR-101 e BR-163 na frente. Por
município a dispersão é grande — os 437 acidentes se distribuem por **334 municípios** — o que
mostra que a concentração crítica está **associada a** rodovia, não a localidade.


> **Ressalva:** o maior evento do período — colisão lateral sentido oposto em TEOFILO OTONI
> (MG), BR-116, com 54 pessoas envolvidas e carga excessiva — levou **37 mortos**. O acidente
> crítico tem ainda perfil de **massa**: mediana de 7 pessoas envolvidas (3 nos demais fatais,
> 2 na base toda) e 3 veículos.


---
## Q12 — Top 5 municípios de Pernambuco em 2024 e 2025.
Quais concentram mais **acidentes fatais**?


In [10]:
pe = df[(df["uf"] == "PE") & (df["ano_acidente"].isin([2024, 2025]))].copy()

q12 = pe.groupby("municipio").agg(
    acidentes=("id", "size"),
    fatais=("acidente_fatal", "sum"),
    mortos=("mortos", "sum"),
)
q12["taxa_letalidade"] = q12["fatais"] / q12["acidentes"]
q12["mortos_por_100"] = q12["mortos"] / q12["acidentes"] * 100

MINIMO = 50
# critério do enunciado: número de acidentes fatais. Desempate por mortos e
# depois por volume, para a ordem ser determinística.
q12_rank = q12[q12["acidentes"] >= MINIMO].sort_values(
    ["fatais", "mortos", "acidentes"], ascending=False)
top5 = q12_rank.head(5)

print(f"Acidentes em PE em 2024-2025 : {len(pe):,}")
print(f"Acidentes fatais             : {int(pe['acidente_fatal'].sum()):,}")
print(f"Mortes                      : {int(pe['mortos'].sum()):,}")
print(f"Municípios com registro      : {pe['municipio'].nunique()}")
print()
print(f"Volume mínimo aplicado: {MINIMO} acidentes no município")
print(f"Municípios excluídos pelo filtro: {len(q12) - len(q12_rank)}")

display(fmt(top5).assign(mortos_por_100=top5["mortos_por_100"].round(1)))

print(f"O top 5 soma {int(top5['fatais'].sum()):,} acidentes fatais "
      f"({top5['fatais'].sum() / pe['acidente_fatal'].sum():.1%} dos fatais de PE), "
      f"{int(top5['mortos'].sum()):,} mortos e {int(top5['acidentes'].sum()):,} acidentes "
      f"({top5['acidentes'].sum() / len(pe):.1%} do total).")

assert len(top5) == 5
assert (top5["acidentes"] >= MINIMO).all()


Acidentes em PE em 2024-2025 : 6,245
Acidentes fatais             : 590
Mortes                      : 649
Municípios com registro      : 90

Volume mínimo aplicado: 50 acidentes no município
Municípios excluídos pelo filtro: 64


,acidentes,fatais,mortos,taxa_letalidade,mortos_por_100
municipio,,,,,
RECIFE,1276,45,47,3.53%,3.7
GARANHUNS,186,24,27,12.90%,14.5
JABOATAO DOS GUARARAPES,465,24,25,5.16%,5.4
CARUARU,409,24,25,5.87%,6.1
PETROLINA,559,23,25,4.11%,4.5


O top 5 soma 140 acidentes fatais (23.7% dos fatais de PE), 149 mortos e 2,895 acidentes (46.4% do total).


In [11]:
# o mesmo conjunto aparece por fatais (critério do enunciado) e por mortos
por_mertos = q12_rank.sort_values(["mortos", "fatais"], ascending=False).head(5)
por_acidentes = q12_rank.sort_values("acidentes", ascending=False).head(5)

print("Top 5 por fatais (enunciado) :", top5.index.tolist())
print("Top 5 por mortos              :", por_mertos.index.tolist())
print("Top 5 por acidentes           :", por_acidentes.index.tolist())
print()
assert set(top5.index) == set(por_mertos.index), \
    "O top 5 deveria ser o mesmo por fatais e por mortos"

print("Maior taxa de letalidade entre os municípios com volume suficiente:")
display(fmt(q12_rank.sort_values("taxa_letalidade", ascending=False).head(5))[
    ["acidentes", "fatais", "mortos", "taxa_letalidade"]
])


Top 5 por fatais (enunciado) : ['RECIFE', 'GARANHUNS', 'JABOATAO DOS GUARARAPES', 'CARUARU', 'PETROLINA']
Top 5 por mortos              : ['RECIFE', 'GARANHUNS', 'JABOATAO DOS GUARARAPES', 'CARUARU', 'PETROLINA']
Top 5 por acidentes           : ['RECIFE', 'PETROLINA', 'JABOATAO DOS GUARARAPES', 'CARUARU', 'IGARASSU']

Maior taxa de letalidade entre os municípios com volume suficiente:


,acidentes,fatais,mortos,taxa_letalidade
municipio,,,,
PESQUEIRA,51,16,19,31.37%
ARARIPINA,58,16,18,27.59%
PAUDALHO,75,16,19,21.33%
BEZERROS,67,14,15,20.90%
ARCOVERDE,82,15,16,18.29%


In [12]:
# o que o filtro de volume deixa de fora — e por que isso importa
excluidos = q12[q12["acidentes"] < MINIMO].sort_values("mortos", ascending=False)
print(f"Municípios abaixo do mínimo de {MINIMO} acidentes: {len(excluidos)}")
display(fmt(excluidos.head(5)[["acidentes", "fatais", "mortos", "taxa_letalidade"]]))

maior_pe = pe.nlargest(1, "mortos").iloc[0]
print()
print(f"Maior evento em PE 2024-2025: {int(maior_pe['mortos'])} mortos — "
      f"{maior_pe['municipio']}, BR-{maior_pe['br']}, "
      f"{maior_pe['data_inversa'].date()}, {maior_pe['tipo_acidente']} / "
      f"{maior_pe['causa_acidente']} ({int(maior_pe['pessoas'])} pessoas envolvidas)")


Municípios abaixo do mínimo de 50 acidentes: 64


,acidentes,fatais,mortos,taxa_letalidade
municipio,,,,
SALOA,13,4,20,30.77%
AGRESTINA,42,11,13,26.19%
CAETES,48,12,12,25.00%
BELO JARDIM,49,9,11,18.37%
RIBEIRAO,39,6,10,15.38%



Maior evento em PE 2024-2025: 16 mortos — SALOA, BR-423, 2025-10-17, Saída de leito carroçável / Velocidade Incompatível (37 pessoas envolvidas)


**Interpretação.** Em PE, no biênio 2024–2025, houve **6.245 acidentes**, 590 fatais e
**649 mortos** — taxa de **9,45%**, acima da global de **7,16%**. O top 5 por **acidentes
fatais** — o critério do enunciado — é **RECIFE (45)**, **GARANHUNS (24)**, **JABOATAO DOS
GUARARAPES (24)**, **CARUARU (24)** e **PETROLINA (23)**: **140 acidentes fatais**, 23,7% dos de
PE. Volume mínimo: **50 acidentes** (90 municípios, 64 excluídos).


**Detalhamento.** O mesmo conjunto sai pelo ranking por **mortos** (149 mortes, 23,0% das
mortes de PE), então a resposta não muda: o ranking é o mesmo pelos dois critérios. O que
separa o grupo é a **intensidade**, não o volume: o resultado está **associado a** Garanhuns
(**12,90%** de taxa de letalidade), enquanto Recife fica em **3,53%**. Entre os municípios que
passam do filtro, os de taxa mais alta são Pesqueira (31,37%), Araripina (27,59%) e Paudalho
(21,33%) — todos com menos de 80 acidentes, o que torna a taxa instável.


> **Ressalva metodológica.** O critério de volume mínimo exclui SALOA, que tem 13 acidentes e
> **20 mortos** — quase todos em um único evento de 17/10/2025 (16 mortos, saída de leito
> carroçável em BR-423, com 37 pessoas envolvidas). Ela não entraria no top 5 por acidentes
> fatais, mas entraria em 6º no ranking por mortos. É a fragilidade do ranking por mortes em
> séries curtas: **um evento pode mover um município inteiro**.


---
## Fechamento do Nível 4

| Questão | Achado |
|---|---|
| Q11 | 437 acidentes (0,205% da base) concentram 9,36% das mortes; **MG lidera** (75 acidentes, 303 mortos); principal causa: contramão (27,0%); colisão frontal 62,9% (razão 2,27); fases sobrerrepresentadas: Amanhecer (1,47) e Pleno dia (1,13) |
| Q12 | Top 5 de PE 2024–2025 por **acidentes fatais**: Recife (45), Garanhuns (24), Jaboatão (24), Caruaru (24), Petrolina (23) — 23,7% dos fatais de PE; mesmo conjunto no ranking por mortos |
